# Calkilo Google Search Console analysis

## tl;dr

- Organic search accelerated sharply: the last 30 days produced **805 clicks from 18,589 impressions**, versus **109 clicks from 3,216 impressions** in the first 30 days. CTR rose from **3.39% to 4.33%**, while impression-weighted position improved from **8.00 to 7.19**.
- Performance is concentrated: `/fa/photo-calorie-calculator/` generated **777 clicks**, or **64.3%** of page-export clicks. Protecting this page matters, but the site needs a second source of growth.
- Three high-volume Persian generic queries generated **3,787 impressions but only 58 clicks (1.53% CTR)**. A scenario of 3.5% CTR at their current rank would add about **75 clicks per 90 days**.
- One suspicious query, `calorie counting recipes for photographers`, produced **2,514 impressions, average position 7.56, and zero clicks**. It is excluded from the opportunity scenario because it appears irrelevant and may distort English-page CTR.
- Mobile drives **94.9% of clicks**. Desktop's lower CTR coincides with a much worse average position (10.60 versus 6.66), so query/country mix should be checked before attributing the gap to UX.


## Context & Methods

This notebook analyzes the Google Search Console Web search export dated July 21, 2026, covering April 20 through July 19, 2026. The decision is where to focus SEO work to grow qualified organic clicks.

### Key Assumptions

- Clicks, impressions, CTR, and average position use Google Search Console definitions.
- Period comparisons use equal-length windows from the daily chart export.
- CTR opportunity is a scenario, not a forecast. It applies a modest rank-band target and excludes the suspicious photographer query.
- Exported query rows are privacy-limited and do not reconcile to top-line totals; query-level conclusions therefore describe the visible query set only.
- Search Console does not include conversions, so traffic quality must be validated with analytics or app-install events.


## Data

### 1. Load and normalize the exports

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

DATA_DIR = Path('/Users/mohammad/Downloads/calkilo.com-Performance-on-Search-2026-07-21')

def load_export(name):
    frame = pd.read_csv(DATA_DIR / f"{name}.csv")
    if "CTR" in frame.columns:
        frame["CTR_numeric"] = frame["CTR"].str.rstrip("%").astype(float) / 100
    return frame

chart = load_export("Chart")
countries = load_export("Countries")
devices = load_export("Devices")
pages = load_export("Pages")
queries = load_export("Queries")
search_appearance = load_export("Search appearance")
filters = load_export("Filters")

chart["Date"] = pd.to_datetime(chart["Date"])
print("Date range:", chart["Date"].min().date(), "to", chart["Date"].max().date())
print("Filters:")
display(filters)


Date range: 2026-04-20 to 2026-07-19
Filters:
        Filter          Value
0  Search type            Web
1         Date  Last 3 months


### 2. Validate grain, completeness, and reconciliation

In [2]:
exports = {
    "Chart": chart,
    "Countries": countries,
    "Devices": devices,
    "Pages": pages,
    "Queries": queries,
    "Search appearance": search_appearance,
}

quality_rows = []
for name, frame in exports.items():
    row = {
        "export": name,
        "rows": len(frame),
        "duplicate_rows": int(frame.duplicated().sum()),
        "null_cells": int(frame.isna().sum().sum()),
    }
    if "Clicks" in frame.columns and len(frame):
        row.update(
            clicks=int(frame["Clicks"].sum()),
            impressions=int(frame["Impressions"].sum()),
            weighted_ctr=frame["Clicks"].sum() / frame["Impressions"].sum(),
        )
    quality_rows.append(row)

quality = pd.DataFrame(quality_rows)
display(quality)
print(
    "Visible query coverage:",
    f"{queries['Clicks'].sum() / chart['Clicks'].sum():.1%} of clicks and "
    f"{queries['Impressions'].sum() / chart['Impressions'].sum():.1%} of impressions",
)


              export  rows  duplicate_rows  null_cells  clicks  impressions  weighted_ctr
0              Chart    91               0           0  1188.0      32826.0      0.036191
1          Countries   152               0           0  1188.0      32826.0      0.036191
2            Devices     3               0           0  1188.0      32826.0      0.036191
3              Pages    91               0           0  1209.0      35420.0      0.034133
4            Queries   479               0           0   664.0      17503.0      0.037936
5  Search appearance     0               0           0     NaN          NaN           NaN
Visible query coverage: 55.9% of clicks and 53.3% of impressions


## Results

### 3. Compare equal 30-day windows

In [3]:
def summarize_period(frame):
    return pd.Series(
        {
            "clicks": int(frame["Clicks"].sum()),
            "impressions": int(frame["Impressions"].sum()),
            "ctr": frame["Clicks"].sum() / frame["Impressions"].sum(),
            "weighted_position": np.average(frame["Position"], weights=frame["Impressions"]),
            "clicks_per_day": frame["Clicks"].mean(),
        }
    )

period_comparison = pd.DataFrame(
    {
        "first_30_days": summarize_period(chart.head(30)),
        "last_30_days": summarize_period(chart.tail(30)),
    }
).T
display(period_comparison)


               clicks  impressions       ctr  weighted_position  clicks_per_day
first_30_days   109.0       3216.0  0.033893           7.998818        3.633333
last_30_days    805.0      18589.0  0.043305           7.189967       26.833333


### 4. Plot weekly search momentum

In [4]:
weekly = chart.assign(position_x_impressions=chart["Position"] * chart["Impressions"]).set_index("Date").resample("W-SUN").agg(
    Clicks=("Clicks", "sum"),
    Impressions=("Impressions", "sum"),
    position_x_impressions=("position_x_impressions", "sum"),
)
weekly["CTR"] = weekly["Clicks"] / weekly["Impressions"]
weekly["Position"] = weekly["position_x_impressions"] / weekly["Impressions"]
display(weekly[["Clicks", "Impressions", "CTR", "Position"]])



            Clicks  Impressions       CTR  Position
Date                                               
2026-04-26      15          345  0.043478  7.698551
2026-05-03      41          925  0.044324  7.785081
2026-05-10      20          848  0.023585  8.433255
2026-05-17      28          876  0.031963  7.781393
2026-05-24      27          892  0.030269  8.123655
2026-05-31      65         1832  0.035480  7.786572
2026-06-07      56         2202  0.025431  8.070527
2026-06-14      63         3410  0.018475  7.129824
2026-06-21      92         3835  0.023990  7.932151
2026-06-28      91         2231  0.040789  8.075527
2026-07-05     201         4172  0.048178  6.991779
2026-07-12     263         5183  0.050743  6.674301
2026-07-19     226         6075  0.037202  7.363984


### 5. Size concentration and segment performance

In [5]:
pages_ranked = pages.sort_values("Clicks", ascending=False).copy()
concentration = pd.Series(
    {
        "top_page_click_share": pages_ranked.iloc[:1]["Clicks"].sum() / pages_ranked["Clicks"].sum(),
        "top_3_page_click_share": pages_ranked.iloc[:3]["Clicks"].sum() / pages_ranked["Clicks"].sum(),
        "top_10_page_click_share": pages_ranked.iloc[:10]["Clicks"].sum() / pages_ranked["Clicks"].sum(),
    }
)
display(concentration.to_frame("share"))
display(pages_ranked.head(10)[["Top pages", "Clicks", "Impressions", "CTR_numeric", "Position"]])

device_summary = devices.copy()
device_summary["click_share"] = device_summary["Clicks"] / device_summary["Clicks"].sum()
device_summary["impression_share"] = device_summary["Impressions"] / device_summary["Impressions"].sum()
display(device_summary[["Device", "Clicks", "Impressions", "CTR_numeric", "Position", "click_share", "impression_share"]])

country_summary = countries.sort_values("Impressions", ascending=False).head(12).copy()
country_summary["click_share"] = country_summary["Clicks"] / countries["Clicks"].sum()
country_summary["impression_share"] = country_summary["Impressions"] / countries["Impressions"].sum()
display(country_summary[["Country", "Clicks", "Impressions", "CTR_numeric", "Position", "click_share", "impression_share"]])


                            share
top_page_click_share     0.642680
top_3_page_click_share   0.844500
top_10_page_click_share  0.981803
                                                 Top pages  Clicks  Impressions  CTR_numeric  Position
0         https://calkilo.com/fa/photo-calorie-calculator/     777        15736       0.0494      6.20
1                                  https://calkilo.com/fa/     177         3903       0.0453      7.43
2            https://calkilo.com/photo-calorie-calculator/      67         5003       0.0134      8.05
3                                     https://calkilo.com/      55         1700       0.0324      9.20
4               https://calkilo.com/fa/ai-calorie-tracker/      53         2075       0.0255      8.06
5         https://calkilo.com/it/photo-calorie-calculator/      29         1795       0.0162      7.70
6                                  https://calkilo.com/it/      15          650       0.0231      7.94
8  https://calkilo.com/it/calcolo-calori

### 6. Rank visible query opportunities without treating irrelevant impressions as upside

In [6]:
def scenario_ctr(position):
    if position <= 3:
        return 0.12
    if position <= 5:
        return 0.07
    if position <= 10:
        return 0.035
    if position <= 20:
        return 0.015
    return 0.005

query_opportunities = queries.copy()
query_opportunities["scenario_ctr"] = query_opportunities["Position"].map(scenario_ctr)
query_opportunities["scenario_click_gain"] = (
    query_opportunities["Impressions"] * query_opportunities["scenario_ctr"]
    - query_opportunities["Clicks"]
).clip(lower=0)
query_opportunities["is_suspicious"] = query_opportunities["Top queries"].eq(
    "calorie counting recipes for photographers"
)

priority_queries = query_opportunities[
    (~query_opportunities["is_suspicious"])
    & (query_opportunities["Impressions"] >= 100)
].sort_values(["scenario_click_gain", "Impressions"], ascending=False)

display(
    priority_queries.head(15)[
        ["Top queries", "Clicks", "Impressions", "CTR_numeric", "Position", "scenario_ctr", "scenario_click_gain"]
    ]
)
print("Scenario gain for non-suspicious queries with >=100 impressions:", round(priority_queries["scenario_click_gain"].sum()))

generic_terms = ["کالری شمار غذا انلاین", "کالری شمار رایگان", "کالری شمار"]
generic_cluster = queries[queries["Top queries"].isin(generic_terms)]
generic_clicks = generic_cluster["Clicks"].sum()
generic_impressions = generic_cluster["Impressions"].sum()
print(
    "Generic Persian cluster:",
    int(generic_clicks), "clicks,",
    int(generic_impressions), "impressions,",
    f"{generic_clicks / generic_impressions:.2%} CTR,",
    f"{generic_impressions * 0.035 - generic_clicks:.0f} clicks to a 3.5% scenario",
)


                     Top queries  Clicks  Impressions  CTR_numeric  Position  scenario_ctr  scenario_click_gain
19                    کالری شمار       6         1001       0.0060      9.71         0.035               29.035
10             کالری شمار رایگان      13         1051       0.0124      7.82         0.035               23.785
2          کالری شمار غذا انلاین      39         1735       0.0225      7.76         0.035               21.725
7   کالری شمار غذا انلاین با عکس      21          442       0.0475      4.93         0.070                9.940
34      برنامه کالری شمار با عکس       2          146       0.0137      7.29         0.035                3.110
40            calcolo calorie ai       1          115       0.0087      6.28         0.035                3.025
41             کالری شمار روزانه       1          115       0.0087      8.07         0.035                3.025
6               محاسبه کالری غذا      23          735       0.0313      5.34         0.035              

## Takeaways

1. **Protect the Persian photo-calorie winner.** It supplies nearly two-thirds of page-export clicks, and its strongest query already earns 11.96% CTR at position 4.90. Avoid broad rewrites that weaken that intent.
2. **Create one clear owner for generic Persian calorie-counter intent.** The three largest generic queries have 3,787 impressions and only 1.53% CTR. Confirm query-to-page overlap in Search Console, then consolidate or differentiate overlapping Persian pages.
3. **Investigate the photographer query before optimizing the English page.** It is the largest visible query by impressions but appears unrelated. A query-by-page export is needed to identify the ranking URL and determine whether the signal is transient, spam-like, or caused by ambiguous page language.
4. **Add product proof, not more generic copy.** The live calculator pages should show a real food-photo example, the returned calorie/macro result, edit flow, screenshots, limitations, and a visible path to try the product.
5. **Prioritize mobile and Persian, then improve Italian.** Mobile supplies 94.9% of clicks; Iran supplies 84.9%. Italy is the clearest secondary market with 3,190 impressions and 58 clicks.
6. **Measure qualified outcomes.** Pair Search Console with landing-page engagement, store-click, install, and signup events. Search clicks alone cannot determine whether an SEO change improves business performance.
